# Infra-Bench: Distributed Imagery Fetch

**Welcome!** Thanks for helping fetch satellite imagery for the Infra-Bench benchmark. This notebook does everything for you — you just need to:

1. Sign in to your Google account (the popup will appear)
2. Create a free Microsoft Planetary Computer account (one-time, takes 2 minutes)
3. Tell the notebook which parquet file Justin assigned you
4. Click **Runtime → Run all**

Total time: **30 min to several hours** depending on your assigned slice. The notebook checkpoints every 50 tiles, so if your Colab session expires you can re-run and it picks up where it left off.

**Questions?** Email Justin (j.guthrie@gmu.edu) or message the lab Slack.

---

## What this notebook does
- Downloads the curation pipeline code from Drive
- Reads your assigned parquet file (a list of geolocated assets — substations, water plants, etc.)
- For each asset, fetches a 600m × 600m satellite image (Sentinel-1 SAR + Sentinel-2 multispectral) from Microsoft Planetary Computer
- Saves each image as a `.npy` file (numpy array) plus a `manifest.json` describing it
- Uploads everything to the shared Drive folder

You don't need to understand any of this. Just run the cells in order.


## Step 1: Mount your Google Drive

This lets the notebook read your assigned parquet from Drive and write your output back to it. **A popup will appear asking you to sign in to Google.** Click through it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2: Tell the notebook your assignment

Edit the cell below to set your assigned parquet. **This is the only thing you need to configure.**

Justin will tell you which parquet you've been assigned. It will be a filename like:
- `asia_water_v1_sample.parquet`
- `europe_transport_v1_sample.parquet`
- etc.

Just paste the filename between the quotes in `MY_ASSIGNED_PARQUET`.

You should not need to change anything else.


In [ ]:
# ============================================================
# your ASSIGNMENT — edit this line:
# ============================================================
MY_ASSIGNED_PARQUET = "PASTE_FILENAME_HERE.parquet"

# ============================================================
# shared resources — do not edit unless Justin says to
# ============================================================
SHARED_DRIVE_ROOT = "/content/drive/MyDrive/infra_fm"
PARQUETS_FOLDER   = f"{SHARED_DRIVE_ROOT}/v1_sample_parquets"      # where parquets live
OUTPUT_FOLDER     = f"{SHARED_DRIVE_ROOT}/v1_imagery_outputs"      # where your output goes
CODE_ZIP          = f"{SHARED_DRIVE_ROOT}/code/infra_fm_curation.zip"

# fetch settings — leave as-is
MODALITIES        = ["sentinel2_ms", "sentinel1"]   # 9-band tiles (7 S2 + 2 S1)
BUFFER_M          = 300                              # 600m x 600m tiles
WORKERS           = 8                                # parallel fetches
CHECKPOINT_EVERY  = 50                               # save progress every 50 tiles

print(f"Assigned: {MY_ASSIGNED_PARQUET}")
print(f"Output:   {OUTPUT_FOLDER}")


## Step 3: Set up your Planetary Computer account

Microsoft Planetary Computer is the satellite data source. You need a free account so the notebook can fetch images on your behalf. **Each lab member uses their own account** — this is what makes parallel fetching work (each account has its own rate limit).

**One-time setup (skip if you already have an account):**

1. Go to **https://planetarycomputer.microsoft.com/account/request** in a new tab
2. Sign in with a Microsoft / GitHub / Google account
3. Fill out the short request form (institutional affiliation, intended use — just say "academic research on critical infrastructure mapping")
4. **Approval is usually instant** but can take up to a day

Once approved, the cell below will work. You don't need to paste any API keys — Planetary Computer uses signed URLs that work automatically once your account is set up.

If you haven't received approval yet, **stop here** and come back when you do.


In [ ]:
# install Planetary Computer client
!pip install -q planetary-computer pystac-client rasterio

# quick test that PC is reachable
import planetary_computer
import pystac_client

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print(f"Planetary Computer reachable: {catalog.title}")


## Step 4: Set up the fetcher

This downloads the project code and gets everything ready. Just run the cell.


In [ ]:
import os, sys, zipfile
from pathlib import Path

EXTRACT_TO = '/content/infra_fm_clean'
CODE_ROOT  = f'{EXTRACT_TO}/infra_fm_code_only'

if not Path(f'{CODE_ROOT}/curation').exists():
    print('Extracting project code...')
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        for member in z.namelist():
            clean_path = member.replace('\\', '/')
            target = os.path.join(EXTRACT_TO, clean_path)
            if clean_path.endswith('/'):
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
    print('Done.')
else:
    print('Code already extracted.')

# curation is a package, so only the repo root goes on the path
sys.path.insert(0, CODE_ROOT)
os.chdir(CODE_ROOT)

from curation.stac_imagery import STACImageryFetcher
from curation.dataset      import DatasetAssembler
print('Ready.')


## Step 5: Load your assignment

This loads the list of assets you need to fetch images for, and tells you roughly how long it will take.


In [ ]:
import pandas as pd
import re

# make sure they actually edited the assignment
if "PASTE_FILENAME_HERE" in MY_ASSIGNED_PARQUET:
    raise ValueError("⚠️ You need to set MY_ASSIGNED_PARQUET in Step 2 before running this cell.")

parquet_path = Path(PARQUETS_FOLDER) / MY_ASSIGNED_PARQUET
if not parquet_path.exists():
    raise FileNotFoundError(
        f"Could not find parquet at {parquet_path}.\n"
        f"Check the filename and verify it's in the shared Drive folder.\n"
        f"Available parquets in folder:\n  " +
        "\n  ".join(sorted(p.name for p in Path(PARQUETS_FOLDER).glob('*.parquet')))
    )

df = pd.read_parquet(parquet_path)
print(f"Loaded {len(df):,} assets from {MY_ASSIGNED_PARQUET}")

# parse region + sector from filename
m = re.match(r"^(?P<region>[a-z-]+?)_(?P<sector>energy|water|transport|telecom)_v1_sample\.parquet$",
             MY_ASSIGNED_PARQUET)
if not m:
    raise ValueError(f"Filename doesn't match expected pattern: {MY_ASSIGNED_PARQUET}")
REGION = m.group("region")
SECTOR = m.group("sector")
print(f"Region: {REGION}")
print(f"Sector: {SECTOR}")

# asset type distribution
print(f"\nAsset type breakdown:")
print(df['asset_type'].value_counts().to_string())

# time estimate (rough — varies with network)
est_minutes = len(df) * 4 / WORKERS / 60  # ~4 sec per tile, parallelized across workers
est_hours   = est_minutes / 60
if est_hours < 1:
    print(f"\nEstimated fetch time: ~{est_minutes:.0f} minutes")
else:
    print(f"\nEstimated fetch time: ~{est_hours:.1f} hours")
print(f"(Checkpoints save every {CHECKPOINT_EVERY} tiles — safe to interrupt and restart)")


## Step 6: Prepare your output folder

Your output goes to a folder named after your assignment, inside the shared Drive folder. If you're restarting after an interruption, the notebook will pick up from where it left off.


In [ ]:
# per-assignment output folder in the shared Drive location.
cell_output_dir = Path(OUTPUT_FOLDER) / f"dataset_{REGION}_{SECTOR}_v1"
cell_output_dir.mkdir(parents=True, exist_ok=True)

# checkpoint lives in the same folder
checkpoint_path = cell_output_dir / "stac_checkpoint.pkl"

print(f"Output folder: {cell_output_dir}")
print(f"Checkpoint:    {checkpoint_path}")

# if a checkpoint exists, show how much is already done
if checkpoint_path.exists():
    import pickle
    with open(checkpoint_path, 'rb') as f:
        ckpt = pickle.load(f)
    n_done = len(ckpt.get('completed_ids', set()))
    print(f"\n📌 Resuming: {n_done:,} of {len(df):,} tiles already fetched ({100*n_done/len(df):.1f}%)")
else:
    print(f"\nStarting fresh.")


## Step 7: Run the fetch

This is the slow part. You can leave this running and come back later. The progress prints every few minutes.

**If your Colab session expires** (usually after ~6-12 hours of inactivity), just re-run this notebook from the top — it will pick up where it left off thanks to the checkpoint file in your Drive folder.

**Tip:** Keep this browser tab open and active so Colab doesn't time out. You can mute the tab and put it in the background.


In [ ]:
# set up the fetcher with your assignment.
fetcher = STACImageryFetcher(
    buffer_m             = BUFFER_M,
    modalities           = MODALITIES,
    temporal_stack       = False,                # single composite, not seasonal stack
    checkpoint_path      = str(checkpoint_path),
    checkpoint_every     = CHECKPOINT_EVERY,
    adaptive_concurrency = True,                 # auto-tune workers based on network
    start_workers        = WORKERS,
    max_workers          = WORKERS * 4,
)

# go.
import time
t0 = time.time()
results = fetcher.fetch_all(df)
elapsed = time.time() - t0

n_ok   = sum(1 for r in results if r.status == "ok")
n_fail = len(results) - n_ok

print(f"\n{'='*50}")
print(f"Fetch complete in {elapsed/60:.1f} minutes")
print(f"  Succeeded: {n_ok:,} / {len(results):,}  ({100*n_ok/max(len(results),1):.1f}%)")
print(f"  Failed:    {n_fail:,}")
print(f"{'='*50}")

if n_fail > 0:
    print(f"\nNote: some failures are normal (clouds, missing scenes, low quality).")
    print(f"Anything above ~50% yield is good.")


## Step 8: Save the final dataset

This packages everything into the standard Infra-Bench dataset format (manifest.json + images folder).


In [ ]:
# only assemble tiles that succeeded
accepted = [r for r in results if r.status == "ok" and r.image is not None]

if accepted:
    assembler = DatasetAssembler(output_dir=str(cell_output_dir))
    summary_df = assembler.assemble(accepted_tiles=accepted, triage_results=None)
    print(f"\n✅ Saved {len(accepted):,} tiles + manifest to:")
    print(f"   {cell_output_dir}")
else:
    print("\n⚠️ No successful tiles to save. Something is wrong — message Justin.")


## Step 9: All done — let Justin know!

Once the cell below runs successfully, you're done. Copy the summary it prints and paste it into a reply to Justin (or in the lab Slack thread).


In [ ]:
# generate a summary message you can paste back to Justin.

if accepted:
    print(f"📩 Paste this into your reply to Justin:\n")
    print(f"{'─'*50}")
    print(f"Assignment:   {MY_ASSIGNED_PARQUET}")
    print(f"Region:       {REGION}")
    print(f"Sector:       {SECTOR}")
    print(f"Targeted:     {len(df):,} assets")
    print(f"Succeeded:    {n_ok:,} ({100*n_ok/len(df):.1f}% yield)")
    print(f"Failed:       {n_fail:,}")
    print(f"Output folder: {cell_output_dir}")
    print(f"{'─'*50}")
else:
    print("⚠️ No tiles were fetched successfully. Please reply to Justin with this notebook's full output.")


---

## Troubleshooting

**Q: I got an error about authentication / 401 / signed URL.**
A: Your Planetary Computer account may not be approved yet. Check https://planetarycomputer.microsoft.com/account/request and wait for approval, then re-run.

**Q: The fetch is really slow.**
A: That's normal at the start while it warms up. Speed usually picks up after the first 200 tiles. If it stays slow (<1 tile/second), your home wifi might be the bottleneck — try running from a faster connection if possible.

**Q: My Colab session got disconnected.**
A: Just re-run the notebook from the top. The checkpoint file in your Drive folder will pick up where you left off. You don't lose progress.

**Q: I got an error about "rate limit" or "429".**
A: Wait 10-15 minutes and re-run. If it keeps happening, message Justin — we may need to space out fetchers' start times.

**Q: A bunch of tiles failed. Should I worry?**
A: 30-50% failure rates are normal (clouds, no good scene available, etc.). Higher than that, message Justin.

**Q: I want to fetch a different parquet (or stop / restart).**
A: Just change `MY_ASSIGNED_PARQUET` in Step 2 and re-run. Each parquet has its own checkpoint so they don't interfere.
